<a href="https://colab.research.google.com/github/mshinno26/UnderstandingAI/blob/main/knn_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [1]:
import csv
import numpy as np # a Python library for calculations
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder

KNN Model

In [2]:
class KNN:
    def __init__(self, k=5, task='classification'):
        self.k = k
        self.task = task

    def fit(self, X_train, y_train): #training function
        self.X_train = X_train
        self.y_train = y_train

    def set_k(self, k):
        self.k = k

    def set_task(self, task):
        self.task = task

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict(self, X_test): #predict function
        predictions = [self.calculate_prediction(x) for x in X_test]
        return np.array(predictions)

    def calculate_prediction(self, x):
        distances = []
        for x_train in self.X_train:
            distance = self.euclidean_distance(x_train, x)
            distances.append(distance)
        #make a new list, with the indices of the k smallest elements of distances
        neighbors = np.argsort(distances)[:self.k]
        targets = []
        closest = []
        for i in neighbors:
            targets.append(self.y_train[i])
            closest.append(distances[i])

        if self.task=='classification':
            #returns the mode
            unique, counts = np.unique(targets, return_counts=True)
            return unique[np.argmax(counts)]
        elif self.task=='regression':
            return int(round(np.mean(targets)))
        # elif self.task=='ordered regression':
        #     #weights are based on order of closeness, not the actual distances
        #     denominator = np.sum(np.array(range(self.k, 0, -1)))
        #     return int(round(np.sum(np.array([(self.k-i)/denominator*targets[i] for i in range(self.k)]))))
        # elif self.task=='weighted regression':
        #     #weights are based directly on the actual distance to the point
        #     denominator = np.sum(distances)
        #     return int(round(np.sum(np.array([(denominator-distances[i])/denominator*targets[i] for i in range(self.k)]))))
        else:
            raise ValueError("Unknown task passed to class KNN(); task must be 'classification', 'regression', or 'weighted regression.")

Functions for data handler

In [3]:
### DONE BY ONE-HOT ENCODER ###
# mcq = {
#     "FAV_CLASS":["Eng","Soc","Math","Sci","Lang","Phys","Art","DTE","Other"],
#     "IS_MATHY":["No","Yes"],
#     "IS_HISPANIC":["No","Yes"],
#     "HASELEC_MATH":["No","Yes"],
#     "MEDIUM_MATH":["App","Pure","Neither"],
#     "HALF_GLASS":["empty","full"],
#     "MEDIUM_NOTE":["Hand","Tab","Type","None"],
#     "ZODIAC":["Aries","Taurus","Gemini","Cancer","Leo","Virgo","Libra","Scorpius","Sagittarius","Capricornus","Aquarius","Pisces"],
#     "SECTION_MATH":["Algebra I (BCP)","Advanced Algebra I/IX","Geometry (BCP)","Geometry","Geometry (H)","Algebra II/Trigonometry (BCP)","Algebra II/Trigonometry","Algebra II/Trigonometry (H)","Pre-Calculus (BCP)","Pre-Calculus with Limits","Advanced Pre-Calculus","Advanced Pre-Calculus (H)","Differential Calculus","AP Calculus AB","AP Calculus BC","None of these"],
#     "RACE":["Asian","White","Black","American/Alaska","Pacific"]
# }

def parse(value, type="int", min=0, max=9999, set_min=False, set_max=False, toss=False):
  try:
    if type=="float":
        value = float(value)
    else:
        value = int(value)
    if set_min:
      if value < min:
        value = min
    if set_max:
      if value > max:
        value = max
    if value == "":
        value = None
    if toss:
        if value > max or value < min:
            value = None

    return value
  except:
    print(f"parse() could not convert {value} to {type}")
    return None

# def convert_height(value):
#     elements = value.split()
#     try:
#         f = int(elements[0])
#         i = int(elements[2])
#         return f*12 + i
#     except:
#         print(f"convert_height() could not convert {value} to number of inches")
#         return None

def convert_height2(value):
    i = 0
    inword = False
    height = ["0","0"]
    try:
        for char in value:
          if char.isdigit():
              inword = False
              height[i] = height[i]+char
          elif not char.isdigit() and not inword:
              inword = True
              i = i+1
        return int(height[0])*12 + int(height[1])
    except:
        print(f"convert_height2() could not convert {value} to number of inches")
        return None

def convert_study(value):
    elements = value.split()
    h = elements[0]
    return h

### DONE WITH ONE-HOT ENCODER ###
"""def enum(value,q):
    opts = mcq[q]
    for i in range(len(opts)):
        if opts[i] in value:
            return i
    return -1

def parse_race(value, min=1, max=5, toss=True):
    num = 0
    l = len(value.split(","))
    for i in range(len(race)):
        if race[i] in value:
            num += pow(2,i)
    if toss:
        if l < min or l > max:
            return None
    return num
"""

'def enum(value,q):\n    opts = mcq[q]\n    for i in range(len(opts)):\n        if opts[i] in value:\n            return i\n    return -1\n\ndef parse_race(value, min=1, max=5, toss=True):\n    num = 0\n    l = len(value.split(","))\n    for i in range(len(race)):\n        if race[i] in value:\n            num += pow(2,i)\n    if toss:\n        if l < min or l > max:\n            return None\n    return num\n'

Data handler

In [4]:
filename = "/content/dataset.csv"
dataset = []
with open(filename) as file:
    reader = csv.DictReader(file)
    for line in reader:
        entry = {}
        for KEY in reader.fieldnames[1:]:
            entry[KEY] = line[KEY]
        dataset.append(entry)

### PARSE DATA - toss values, turn strings into ints, one-hot encoding ###
flags = []
for line in dataset:
    for KEY in line:
        if KEY == 'USER_ID':
            line[KEY] = parse(line[KEY])
        elif KEY == 'LEN_BOOK':
            line[KEY] = parse(line[KEY], max=2000, toss=True)
        elif KEY == 'SCREENTIME':
            line[KEY] = parse(line[KEY], min=0, max=16, toss=True)
        elif KEY == 'INTRO_EXTRO':
            line[KEY] = parse(line[KEY])
        elif KEY == 'HOURS_SLEEP':
            line[KEY] = parse(line[KEY], type="float")
        elif KEY == 'TESTSCORE_MATH':
            line[KEY] = parse(line[KEY], min = 5, toss=True)
        elif KEY == 'AGE':
            line[KEY] = parse(line[KEY], min = 12, max = 19, toss=True)
        elif KEY == 'HEIGHT':
            new = convert_height2(line[KEY])
            line[KEY] = parse(new, min = 56, max = 82, toss=True)
        elif KEY == 'HOURS_STUDY':
            new = convert_study(line[KEY])
            line[KEY] = parse(new,type="float")
        ### DONE WITH ONE-HOT ENCODER ###
        # elif KEY == 'RACE':
        #     line[KEY] = parse_race(line[KEY])
        # elif KEY in mcq.keys():
        #     new = enum(line[KEY],KEY)
        #     line[KEY] = parse(new)
    if None in line.values():
        flags.append(line)

for flag in flags:
    dataset.remove(flag)

parse() could not convert  to int
parse() could not convert  to int
parse() could not convert  to int
parse() could not convert  to float
parse() could not convert 0.1 to int


Turn dicts into dataframes

In [6]:
# data (as pandas dataframes)
keys = ["LEN_BOOK","HOURS_SLEEP","HOURS_STUDY","SECTION_MATH"]
target = "SCREENTIME"
features = {key:[] for key in keys}
targets = {target:[]}
for row in dataset:
  for key in keys:
    features[key].append(row[key])
  targets[target].append(row[target])
X = pd.DataFrame(features)
y = pd.DataFrame(targets)

# Encode MCQ columns:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
catcols = X.select_dtypes(include='object').columns

for column in catcols:
    X[column] = X[column].astype(str)

if not catcols.empty:
    encoded = encoder.fit_transform(X[[column]])
    frame = pd.DataFrame(encoded,
                      columns=encoder.get_feature_names_out([column]),
                      index=X.index)
    X = pd.concat([X.drop(column, axis=1), frame], axis=1)

Finding best test/train, k, and task

In [8]:
# for finding best k value/task:

knn = KNN()
trials = []
accuracies = []
tasks = ['classification', 'regression', 'ordered regression', 'weighted regression']
knn.set_task(tasks[1])

# test a range of k values for each task
# for j in range(1, 50):
for x in range(5,30):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=x/100, random_state=42)
  knn.fit(X_train.values, y_train.values)
  for k in range(1,70):
    knn.set_k(k)
    # Make predictions using the already encoded test data
    trials.append(str(k) + ", " + str(x))
    try:
      y_pred = knn.predict(X_test.values)
      accuracy_custom = accuracy_score(y_test, y_pred)
      accuracies.append(accuracy_custom)
    except:
      break

In [9]:
# Find most accurate tasks and k-values
accurate_indices = list(np.argsort(accuracies))
accurate_indices.reverse()
for i in accurate_indices[:100]:
    print(trials[i])
    print(accuracies[i])

2, 5
0.4
29, 6
0.3333333333333333
59, 6
0.3333333333333333
57, 6
0.3333333333333333
28, 27
0.3333333333333333
33, 6
0.3333333333333333
31, 6
0.3333333333333333
34, 6
0.3333333333333333
32, 6
0.3333333333333333
10, 6
0.3333333333333333
35, 6
0.3333333333333333
62, 6
0.3333333333333333
11, 6
0.3333333333333333
14, 6
0.3333333333333333
2, 6
0.3333333333333333
24, 6
0.3333333333333333
61, 6
0.3333333333333333
5, 6
0.3333333333333333
9, 6
0.3333333333333333
6, 6
0.3333333333333333
7, 6
0.3333333333333333
8, 6
0.3333333333333333
60, 6
0.3333333333333333
30, 6
0.3333333333333333
36, 6
0.3333333333333333
58, 6
0.3333333333333333
38, 6
0.3333333333333333
37, 6
0.3333333333333333
56, 6
0.3333333333333333
48, 25
0.3181818181818182
29, 25
0.3181818181818182
27, 21
0.3157894736842105
20, 21
0.3157894736842105
24, 21
0.3157894736842105
20, 18
0.3125
48, 26
0.30434782608695654
29, 26
0.30434782608695654
24, 22
0.3
29, 22
0.3
19, 22
0.3
22, 22
0.3
23, 22
0.3
27, 22
0.3
20, 19
0.29411764705882354
30, 2

Hardcoded test/train, k, and task

In [10]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# How many feautres (cols) and entries (rows) are in the training data?
print("Training features shape:", X_train.shape)

# How many labels/answers do we have to train our model on?
print("Training labels shape:", y_train.shape)

# Same info as above, for texting the accuracy of our model.
print("Testing features shape:", X_test.shape)
print("Testing labels shape:", y_test.shape)

Training features shape: (65, 17)
Training labels shape: (65, 1)
Testing features shape: (22, 17)
Testing labels shape: (22, 1)


In [11]:
tasks = ['classification', 'regression', 'ordered regression', 'weighted regression']

knn = KNN()
knn.fit(X_train.values, y_train.values)
knn.set_k(29)
knn.set_task(tasks[1])
y_pred = knn.predict(X_test.values)

In [12]:
# Calculate accuracy
accuracy_custom = accuracy_score(y_test, y_pred)
print("Custom Accuracy:", accuracy_custom)

Custom Accuracy: 0.3181818181818182


SKLearn comparison

In [13]:
# Create and train the KNN model, to compare our accuracy to that of SKLearn's model
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(X_train, y_train)

# Make predictions on the test set
y_pred = knn.predict(X_test)

# Calculate accuracy
accuracy_sklearn = accuracy_score(y_test, y_pred)
print("Sklearn Accuracy:", accuracy_sklearn)

Sklearn Accuracy: 0.13636363636363635


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
